In [ ]:
import pandas as pd
import numpy as np
import Bio
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Restriction import AllEnzymes
import itertools
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Load Required Data

In [ ]:
# Load methylation sensitivity data
df_methylation = pd.read_csv('./utils/output/methylation_check.csv')

# Load NEB enzyme characteristics
df_neb = pd.read_csv('./utils/output/neb_buffer_activity_cleaned.csv')

# Load plasmid compatibility data
df_plasmid_check = pd.read_csv('./utils/output/plasmid_digest_check.csv', index_col=0)

# Load silent mutation data (Site II candidates)
df_silent_mutation = pd.read_csv('./utils/output/restriction_enzyme_slient_mutation.csv')

# Load seamless insert data (Site I candidates)
df_seamless_insert = pd.read_csv('./utils/output/restriction_enzyme_seamless_insert.csv')

# Load orthogonality data
df_orthogonality = pd.read_csv('./utils/output/orthogonality.csv')

print(f"Loaded data:")
print(f"  Methylation check: {df_methylation.shape}")
print(f"  NEB characteristics: {df_neb.shape}")
print(f"  Plasmid compatibility: {df_plasmid_check.shape}")
print(f"  Silent mutation options: {df_silent_mutation.shape}")
print(f"  Seamless insert options: {df_seamless_insert.shape}")
print(f"  Orthogonality matrix: {df_orthogonality.shape}")

## 2. Helper Functions

In [ ]:
def check_methylation_compatible(enzyme_name, df_methylation):
    """Check if enzyme is not sensitive to 6mA/5mC methylation (DH5α compatible)"""
    dh5a_compatible = set(
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'enzyme'].dropna().tolist() +
        df_methylation.loc[~df_methylation['6mA_5mC_sensitive'], 'prototype'].dropna().tolist()
    )
    return enzyme_name in dh5a_compatible

def check_neb_quality(enzyme_name, df_neb):
    """Check if enzyme has good NEB characteristics"""
    if enzyme_name not in df_neb['enzyme'].values:
        return False
    
    enzyme_data = df_neb[df_neb['enzyme'] == enzyme_name].iloc[0]
    
    # Check ligation efficiency is not low
    if enzyme_data['ligation_efficiencies'] == 'low':
        return False
    
    # Check no star activity
    if enzyme_data['star_activity'] == True:
        return False
    
    return True

def is_type_iis(enzyme_name):
    """Check if enzyme is Type IIS (cuts outside recognition sequence)"""
    try:
        enzyme = getattr(Bio.Restriction, enzyme_name)
        # Type IIS enzymes have cut sites outside recognition sequence
        # Check if fst5 or fst3 is outside the site length
        site_length = len(enzyme.site)
        return enzyme.fst5 < 0 or enzyme.fst5 > site_length or enzyme.fst3 < 0 or enzyme.fst3 > site_length
    except:
        return False

def get_ovhg_pattern(enzyme_name):
    """Get overhang pattern (length and direction)"""
    try:
        enzyme = getattr(Bio.Restriction, enzyme_name)
        return enzyme.ovhg  # Returns overhang length with sign
    except:
        return None

def check_orthogonality(enzyme1, enzyme2, df_orthogonality, min_score=2):
    """Check if two enzymes are orthogonal (non-complementary sticky ends)"""
    # Look up orthogonality score
    result = df_orthogonality[
        ((df_orthogonality['re1'] == enzyme1) & (df_orthogonality['re2'] == enzyme2)) |
        ((df_orthogonality['re1'] == enzyme2) & (df_orthogonality['re2'] == enzyme1))
    ]
    
    if len(result) > 0:
        return result.iloc[0]['orthogonality'] >= min_score
    return False

def check_plasmid_compatible(enzyme_name, plasmid_name, df_plasmid_check):
    """Check if enzyme does not cut plasmid backbone"""
    if enzyme_name in df_plasmid_check.index and plasmid_name in df_plasmid_check.columns:
        return df_plasmid_check.loc[enzyme_name, plasmid_name] == True
    return False

## 3. Build Enzyme Pools

In [ ]:
# Get basic enzyme information from Bio.Restriction
from Bio.Restriction.Restriction_Dictionary import rest_dict

df_all_enzymes = pd.DataFrame(rest_dict).T
df_all_enzymes['name'] = df_all_enzymes.index

# Filter for basic requirements: commercially available, defined sequence, 2-5bp overhang
df_basic = df_all_enzymes[
    (df_all_enzymes['suppl'].str.len() > 0) &  # Commercially available
    (df_all_enzymes['site'].str.match('^[ATCG]+$')) &  # No ambiguous bases
    (df_all_enzymes['ovhgseq'].str.match('^[ATCG]+$')) &  # Defined overhang
    (df_all_enzymes['ovhg'].isin([-5, -4, -3, -2, 2, 3, 4, 5]))  # 2-5bp overhang
].copy()

print(f"Enzymes passing basic requirements: {len(df_basic)}")

In [ ]:
# Build Site I pool: methylation compatible, no star activity
site_i_candidates = []
for enzyme in df_basic['name']:
    if (check_methylation_compatible(enzyme, df_methylation) and 
        check_neb_quality(enzyme, df_neb)):
        site_i_candidates.append(enzyme)

print(f"Site I candidates: {len(site_i_candidates)}")

# Build Site II pool: same as Site I + allows silent mutation
site_ii_candidates = df_silent_mutation['name'].unique().tolist()
site_ii_candidates = [e for e in site_ii_candidates if e in site_i_candidates]

print(f"Site II candidates: {len(site_ii_candidates)}")

# Build Site III pool: Type IIS enzymes
site_iii_candidates = []
for enzyme in df_basic['name']:
    if is_type_iis(enzyme) and check_neb_quality(enzyme, df_neb):
        site_iii_candidates.append(enzyme)

print(f"Site III candidates (Type IIS): {len(site_iii_candidates)}")

## 4. Generate All Valid Site Combinations

In [ ]:
# Find all valid three-site combinations
valid_combinations = []

print("Generating valid site combinations...")
print(f"Checking {len(site_i_candidates)} × {len(site_ii_candidates)} × {len(site_iii_candidates)} combinations")

for site_i in tqdm(site_i_candidates, desc="Site I"):
    ovhg_i = get_ovhg_pattern(site_i)
    
    for site_ii in site_ii_candidates:
        ovhg_ii = get_ovhg_pattern(site_ii)
        
        # Check Site I and II have different ovhg pattern OR are orthogonal
        if ovhg_i == ovhg_ii:
            # Same pattern, check orthogonality
            if not check_orthogonality(site_i, site_ii, df_orthogonality, min_score=2):
                continue
        
        for site_iii in site_iii_candidates:
            ovhg_iii = get_ovhg_pattern(site_iii)
            
            # Check Site II and III have same ovhg pattern
            if ovhg_ii != ovhg_iii:
                continue
            
            # All conditions met, add to valid combinations
            valid_combinations.append({
                'site_i': site_i,
                'site_ii': site_ii,
                'site_iii': site_iii,
                'ovhg_i': ovhg_i,
                'ovhg_ii': ovhg_ii,
                'ovhg_iii': ovhg_iii
            })

df_combinations = pd.DataFrame(valid_combinations)
print(f"\nTotal valid combinations: {len(df_combinations)}")

## 5. Check Plasmid Compatibility - Generate df1

In [ ]:
# Define plasmids to check
plasmids = ['pGEX-4T-1', 'pMAL-c5X', 'pET-21a(+)', 'pET-28a(+)', 
            'pET-28a(+)_start_codon', 'pCold_I', 'pUC18', 'pQE-3']

# Check plasmid compatibility for each combination
print("Checking plasmid compatibility...")

for plasmid in plasmids:
    df_combinations[f'{plasmid}_compatible'] = df_combinations.apply(
        lambda row: (
            check_plasmid_compatible(row['site_i'], plasmid, df_plasmid_check) and
            check_plasmid_compatible(row['site_ii'], plasmid, df_plasmid_check)
        ),
        axis=1
    )

# Create df1
df1 = df_combinations.copy()

# Add summary column: which plasmids are compatible
df1['compatible_plasmids'] = df1.apply(
    lambda row: [p for p in plasmids if row[f'{p}_compatible']],
    axis=1
)

df1['num_compatible_plasmids'] = df1['compatible_plasmids'].apply(len)

print(f"\ndf1 created with {len(df1)} combinations")
print(f"Combinations compatible with at least one plasmid: {(df1['num_compatible_plasmids'] > 0).sum()}")

# Save df1
df1.to_csv('./output/hurdler_three_site_combinations_df1.csv', index=False)
print("Saved to: ./output/hurdler_three_site_combinations_df1.csv")

In [ ]:
# Display summary statistics
print("\n=== Summary Statistics ===")
print(f"Total site combinations: {len(df1)}")
print(f"\nPlasmid compatibility:")
for plasmid in plasmids:
    count = df1[f'{plasmid}_compatible'].sum()
    print(f"  {plasmid}: {count} combinations ({count/len(df1)*100:.1f}%)")

print(f"\nUnique enzymes used:")
print(f"  Site I: {df1['site_i'].nunique()}")
print(f"  Site II: {df1['site_ii'].nunique()}")
print(f"  Site III: {df1['site_iii'].nunique()}")

df1.head(10)

## 6. Add 3mer AA Sequences - Generate df2

In [ ]:
# Load 3mer AA data for Site I (seamless insert)
df_site_i_aa = df_seamless_insert[['name', 're_site_shifted_tl', 're_site_shifted', 
                                     'frame_shift', 'codon_usage']].copy()
df_site_i_aa.columns = ['site_i', 'site_i_3mer_aa', 'site_i_dna', 'site_i_frame', 'site_i_codon_usage']

# Load 3mer AA data for Site II (silent mutation)
df_site_ii_aa = df_silent_mutation[['name', 're_site_shifted_tl', 're_site_shifted', 
                                     're_site_mutate_shifted', 'frame_shift', 'codon_usage_mutate']].copy()
df_site_ii_aa.columns = ['site_ii', 'site_ii_3mer_aa', 'site_ii_dna', 'site_ii_dna_mutated', 
                          'site_ii_frame', 'site_ii_codon_usage']

print(f"Site I 3mer AA entries: {len(df_site_i_aa)}")
print(f"Site II 3mer AA entries: {len(df_site_ii_aa)}")
print(f"Unique Site I 3mer AA: {df_site_i_aa['site_i_3mer_aa'].nunique()}")
print(f"Unique Site II 3mer AA: {df_site_ii_aa['site_ii_3mer_aa'].nunique()}")

In [ ]:
# Create df2 by joining df1 with 3mer AA data
print("Creating df2 with 3mer AA sequences...")

# First merge with Site I data
df2 = df1.merge(df_site_i_aa, on='site_i', how='left')

# Then merge with Site II data
df2 = df2.merge(df_site_ii_aa, on='site_ii', how='left')

# Remove combinations without 3mer AA data
df2 = df2.dropna(subset=['site_i_3mer_aa', 'site_ii_3mer_aa'])

print(f"\ndf2 created with {len(df2)} combinations")
print(f"Unique (Site I enzyme, 3mer AA) pairs: {df2[['site_i', 'site_i_3mer_aa']].drop_duplicates().shape[0]}")
print(f"Unique (Site II enzyme, 3mer AA) pairs: {df2[['site_ii', 'site_ii_3mer_aa']].drop_duplicates().shape[0]}")

# Save df2
df2.to_csv('./output/hurdler_three_site_combinations_df2.csv', index=False)
print("Saved to: ./output/hurdler_three_site_combinations_df2.csv")

In [ ]:
# Display sample of df2
print("\n=== Sample of df2 ===")
df2[['site_i', 'site_i_3mer_aa', 'site_ii', 'site_ii_3mer_aa', 'site_iii', 
     'ovhg_i', 'ovhg_ii', 'compatible_plasmids']].head(10)

## 7. Query Function: Find Sites by 3mer AA and Plasmid

In [ ]:
def find_hurdler_sites(site_i_3mer_aa, site_ii_3mer_aa, plasmid, df2=df2):
    """
    Find all valid three-site combinations for given 3mer AA sequences and plasmid.
    
    Parameters:
    -----------
    site_i_3mer_aa : str
        3-amino acid sequence for Site I (seamless insert)
    site_ii_3mer_aa : str
        3-amino acid sequence for Site II (silent mutation)
    plasmid : str
        Plasmid name (e.g., 'pET-28a(+)')
    df2 : DataFrame
        The df2 DataFrame containing all combinations
    
    Returns:
    --------
    DataFrame with matching combinations, or None if no matches found
    """
    # Check if plasmid is valid
    plasmid_col = f'{plasmid}_compatible'
    if plasmid_col not in df2.columns:
        print(f"Error: Plasmid '{plasmid}' not found. Available plasmids:")
        available = [col.replace('_compatible', '') for col in df2.columns if col.endswith('_compatible')]
        print(f"  {', '.join(available)}")
        return None
    
    # Filter for matching 3mer AA sequences and plasmid compatibility
    results = df2[
        (df2['site_i_3mer_aa'] == site_i_3mer_aa) &
        (df2['site_ii_3mer_aa'] == site_ii_3mer_aa) &
        (df2[plasmid_col] == True)
    ].copy()
    
    if len(results) == 0:
        print(f"No valid combinations found for:")
        print(f"  Site I 3mer AA: {site_i_3mer_aa}")
        print(f"  Site II 3mer AA: {site_ii_3mer_aa}")
        print(f"  Plasmid: {plasmid}")
        return None
    
    print(f"Found {len(results)} valid combination(s):")
    print(f"  Site I 3mer AA: {site_i_3mer_aa}")
    print(f"  Site II 3mer AA: {site_ii_3mer_aa}")
    print(f"  Plasmid: {plasmid}")
    print(f"\nDetails:")
    
    for idx, row in results.iterrows():
        print(f"\n  Combination {idx}:")
        print(f"    Site I:   {row['site_i']} (ovhg: {row['ovhg_i']})")
        print(f"    Site II:  {row['site_ii']} (ovhg: {row['ovhg_ii']}, allows silent mutation)")
        print(f"    Site III: {row['site_iii']} (ovhg: {row['ovhg_iii']}, Type IIS)")
    
    return results[['site_i', 'site_i_3mer_aa', 'site_i_dna', 'site_i_frame',
                    'site_ii', 'site_ii_3mer_aa', 'site_ii_dna', 'site_ii_dna_mutated', 'site_ii_frame',
                    'site_iii', 'ovhg_i', 'ovhg_ii', 'ovhg_iii']]

## 8. Test Query Function

In [ ]:
# Test with some example 3mer AA sequences
# Get some common 3mer AA from the data
example_site_i_aa = df2['site_i_3mer_aa'].value_counts().head(5).index.tolist()
example_site_ii_aa = df2['site_ii_3mer_aa'].value_counts().head(5).index.tolist()

print("=== Top 5 most common Site I 3mer AA ===")
for aa in example_site_i_aa:
    count = (df2['site_i_3mer_aa'] == aa).sum()
    print(f"  {aa}: {count} combinations")

print("\n=== Top 5 most common Site II 3mer AA ===")
for aa in example_site_ii_aa:
    count = (df2['site_ii_3mer_aa'] == aa).sum()
    print(f"  {aa}: {count} combinations")

In [ ]:
# Test the function with an example
print("=== Testing Query Function ===")
print("\nExample 1:")
result1 = find_hurdler_sites(
    site_i_3mer_aa=example_site_i_aa[0],
    site_ii_3mer_aa=example_site_ii_aa[0],
    plasmid='pET-28a(+)'
)

if result1 is not None:
    display(result1)

## 9. Create Lookup Tables for Quick Query

In [ ]:
# Create a lookup table: (Site I 3mer AA, Site II 3mer AA, Plasmid) -> list of valid combinations
print("Creating lookup tables...")

# Group by 3mer AA sequences and plasmid
lookup_data = []

for plasmid in plasmids:
    plasmid_col = f'{plasmid}_compatible'
    df_plasmid_subset = df2[df2[plasmid_col] == True]
    
    grouped = df_plasmid_subset.groupby(['site_i_3mer_aa', 'site_ii_3mer_aa'])
    
    for (aa_i, aa_ii), group in grouped:
        lookup_data.append({
            'site_i_3mer_aa': aa_i,
            'site_ii_3mer_aa': aa_ii,
            'plasmid': plasmid,
            'num_combinations': len(group),
            'site_i_enzymes': group['site_i'].unique().tolist(),
            'site_ii_enzymes': group['site_ii'].unique().tolist(),
            'site_iii_enzymes': group['site_iii'].unique().tolist()
        })

df_lookup = pd.DataFrame(lookup_data)
df_lookup.to_csv('./output/hurdler_3mer_aa_lookup.csv', index=False)

print(f"Lookup table created with {len(df_lookup)} entries")
print(f"Unique (3mer AA I, 3mer AA II) pairs across all plasmids: {df_lookup[['site_i_3mer_aa', 'site_ii_3mer_aa']].drop_duplicates().shape[0]}")
print("Saved to: ./output/hurdler_3mer_aa_lookup.csv")

In [ ]:
# Show sample of lookup table
print("\n=== Sample of Lookup Table ===")
df_lookup.head(10)

## 10. Statistics and Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Plot 1: Number of valid combinations per plasmid
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count combinations per plasmid
plasmid_counts = {}
for plasmid in plasmids:
    plasmid_counts[plasmid] = df1[f'{plasmid}_compatible'].sum()

# Plot bar chart
axes[0].barh(list(plasmid_counts.keys()), list(plasmid_counts.values()))
axes[0].set_xlabel('Number of Valid Site Combinations')
axes[0].set_title('Valid Three-Site Combinations per Plasmid')

# Plot 2: Number of 3mer AA pairs per plasmid
plasmid_3mer_counts = df_lookup.groupby('plasmid')['num_combinations'].sum().sort_values(ascending=True)
axes[1].barh(plasmid_3mer_counts.index, plasmid_3mer_counts.values)
axes[1].set_xlabel('Total 3mer AA Combinations')
axes[1].set_title('Total 3mer AA Combinations per Plasmid')

plt.tight_layout()
plt.savefig('./output/hurdler_plasmid_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Distribution of overhang patterns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Site I overhang distribution
df1['ovhg_i'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Overhang Length (bp)')
axes[0].set_ylabel('Count')
axes[0].set_title('Site I Overhang Distribution')
axes[0].tick_params(axis='x', rotation=0)

# Site II overhang distribution
df1['ovhg_ii'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_xlabel('Overhang Length (bp)')
axes[1].set_ylabel('Count')
axes[1].set_title('Site II Overhang Distribution')
axes[1].tick_params(axis='x', rotation=0)

# Site III overhang distribution
df1['ovhg_iii'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='mediumseagreen')
axes[2].set_xlabel('Overhang Length (bp)')
axes[2].set_ylabel('Count')
axes[2].set_title('Site III Overhang Distribution')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('./output/hurdler_overhang_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 4: Top enzymes used in each site
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Top Site I enzymes
df1['site_i'].value_counts().head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 15 Site I Enzymes')

# Top Site II enzymes
df1['site_ii'].value_counts().head(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 15 Site II Enzymes')

# Top Site III enzymes
df1['site_iii'].value_counts().head(15).plot(kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_xlabel('Frequency')
axes[2].set_title('Top 15 Site III Enzymes')

plt.tight_layout()
plt.savefig('./output/hurdler_top_enzymes.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Final Summary Report

In [ ]:
print("="*80)
print("HURDLER THREE-SITE COMBINATION ANALYSIS - FINAL SUMMARY")
print("="*80)

print("\n1. ENZYME POOLS")
print(f"   - Site I candidates (methylation insensitive): {len(site_i_candidates)}")
print(f"   - Site II candidates (allows silent mutation): {len(site_ii_candidates)}")
print(f"   - Site III candidates (Type IIS): {len(site_iii_candidates)}")

print("\n2. VALID COMBINATIONS (df1)")
print(f"   - Total three-site combinations: {len(df1):,}")
print(f"   - Combinations with at least one compatible plasmid: {(df1['num_compatible_plasmids'] > 0).sum():,}")

print("\n3. PLASMID COMPATIBILITY")
for plasmid in plasmids:
    count = df1[f'{plasmid}_compatible'].sum()
    print(f"   - {plasmid}: {count:,} combinations")

print("\n4. 3MER AA COMBINATIONS (df2)")
print(f"   - Total site-3mer AA combinations: {len(df2):,}")
print(f"   - Unique Site I 3mer AA sequences: {df2['site_i_3mer_aa'].nunique():,}")
print(f"   - Unique Site II 3mer AA sequences: {df2['site_ii_3mer_aa'].nunique():,}")
print(f"   - Unique (Site I AA, Site II AA) pairs: {df2[['site_i_3mer_aa', 'site_ii_3mer_aa']].drop_duplicates().shape[0]:,}")

print("\n5. OUTPUT FILES")
print(f"   - ./output/hurdler_three_site_combinations_df1.csv")
print(f"   - ./output/hurdler_three_site_combinations_df2.csv")
print(f"   - ./output/hurdler_3mer_aa_lookup.csv")
print(f"   - ./output/hurdler_plasmid_statistics.png")
print(f"   - ./output/hurdler_overhang_distribution.png")
print(f"   - ./output/hurdler_top_enzymes.png")

print("\n6. QUERY FUNCTION")
print(f"   - Use find_hurdler_sites(site_i_3mer_aa, site_ii_3mer_aa, plasmid)")
print(f"   - Returns valid three-site combinations for given 3mer AA sequences and plasmid")

print("\n" + "="*80)
print("Analysis complete!")
print("="*80)